# Kafka events in DuckLake

Run `just up-kafka`, `just consume-events`, then `just produce-events --count 12 --duplicate-every 4` in separate terminals. The table appears after the consumer flushes its first batch, by default after 50 messages or 5 seconds. Rerun the query cells to see later batches.


In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from ducklake_playground import DuckLakeEngine, load_config

config = load_config(repo_root / "config.yaml")
engine = DuckLakeEngine()
engine.setup(config, "local")
con = engine.connection
catalog = engine.catalog_name
table = engine.qualified_table("kafka_events")
print(f"Connected to {table}. Data path: {engine.data_path}")


## Rows and distinct event IDs

Duplicates and full Kafka replays should leave these two numbers equal.


In [ ]:
con.execute(f"""
    SELECT COUNT(*) AS rows, COUNT(DISTINCT id) AS distinct_ids
    FROM {table}
""").pl()


## Latest events


In [ ]:
con.execute(f"""
    SELECT id, timestamp_col, event_date, varchar_col, int64_col
    FROM {table}
    ORDER BY timestamp_col DESC
    LIMIT 20
""").pl()


## Recent DuckLake snapshots

Each committed consumer batch adds a snapshot.


In [ ]:
con.execute(f"""
    SELECT snapshot_id, snapshot_time
    FROM {catalog}.snapshots()
    ORDER BY snapshot_id DESC
    LIMIT 10
""").pl()
